In [ ]:
# velo reward computation for rollouts on game24


"""Setup: read eval_rollout.jsonl. No model, no GPU.

`script/run_game24_one.py --score-vt` (default) augments each rollout row
with R_T, R_per_token, and a fixed-grid cumR_resampled. All three figures
below run from those fields plus the original (numbers, correct, n_tokens,
completion, global_step) columns.
"""
import json
from pathlib import Path
from math import comb

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Point this at the output directory produced by script/run_game24_one.py
RUN_DIR  = Path("logs/game24_sweep_exp4/len512/Qwen__Qwen3-4B")
EVAL_LOG = RUN_DIR / "eval_rollout.jsonl"
assert EVAL_LOG.exists(), f"missing {EVAL_LOG}; run script/run_game24_one.py first"

rows = [json.loads(l) for l in EVAL_LOG.read_text().splitlines() if l.strip()]
eval_df = pd.DataFrame(rows)
eval_df["key"] = eval_df["numbers"].apply(lambda x: tuple(sorted(x)))

has_vt = "R_T" in eval_df.columns and eval_df["R_T"].notna().any()
print(f"{len(eval_df)} eval rollouts across "
      f"{eval_df.global_step.nunique()} eval cycles "
      f"(global_step ∈ {sorted(eval_df.global_step.unique().tolist())})")
print(f"R_T fields present: {has_vt}")
eval_df.head(3)

576 eval rollouts across 1 eval cycles (global_step ∈ [1200])
per-ref vt fields present: True


,step,global_step,idx,numbers,expr,correct,refs,own_ref_idx,best_ref_idx,R_T_per_ref,R_per_token_per_ref,cumR_resampled_per_ref,key
0,108,1200,0,"[3, 5, 8, 9]",,False,"[((9*3)-8)+5, (3*9)-(8-5), 5-(8-(9*3)), 5-(8-(...",-1,15,"[-20.400177001953125, -15.942626953125, -14.65...","[-0.0398440957069397, -0.031137943267822266, -...","[[3.7933883666992188, -19.18015343251854, -20....","(3, 5, 8, 9)"
1,108,1200,1,"[3, 5, 8, 9]",,False,"[((9*3)-8)+5, (3*9)-(8-5), 5-(8-(9*3)), 5-(8-(...",-1,12,"[-21.677169799804688, -22.03948974609375, -25....","[-0.04233822226524353, -0.043045878410339355, ...","[[3.7933883666992188, -19.18015343251854, -20....","(3, 5, 8, 9)"
2,108,1200,2,"[3, 5, 8, 9]",,False,"[((9*3)-8)+5, (3*9)-(8-5), 5-(8-(9*3)), 5-(8-(...",-1,16,"[-19.899612426757812, -1.81402587890625, -6.15...","[-0.03886643052101135, -0.0035430192947387695,...","[[3.7933883666992188, -19.18015343251854, -20....","(3, 5, 8, 9)"


In [2]:
scorer_model = "Qwen/Qwen3-0.6B"
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok_src = scorer_model
# =============================
# Load scorer model & tokenizer
# =============================
tokenizer = AutoTokenizer.from_pretrained(tok_src)
device = (f"cuda:{torch.cuda.current_device()}"
                if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
# scorer = AutoModelForCausalLM.from_pretrained(
#     scorer_model, dtype=dtype
# ).to(device).eval()

# ==================================================
# Compute Velocity reward on a specific step & idx
# ==================================================
import ast
from script.rescore_vt import enumerate_solutions, to_chat

# AST-canonicalize so '(5*5- (2 - 1))' == '(5*5)-(2-1)'.
def canon(expr: str) -> str:
    try:    return ast.dump(ast.parse(expr, mode="eval").body)
    except: return expr

step, idx = 1200, 0
select_rows = [r for r in rows if r['global_step'] == step and r['idx'] == idx and r['expr']]
assert select_rows, f"No row found for step {step} and idx {idx}"
row = select_rows[0]

sols = enumerate_solutions(tuple(row["numbers"]))
puzzle = {"numbers": list(row["numbers"]), "solutions": sols}
prompt = tokenizer.apply_chat_template(
    to_chat(puzzle)["prompt"], tokenize=False, add_generation_prompt=True)

# refs = canonical sols, plus row['expr'] iff AST-distinct from all of them.
refs = list(sols)
own_canon = canon(row["expr"])
canon_set = {canon(s) for s in sols}
own_idx = next((i for i, s in enumerate(sols) if canon(s) == own_canon), None)
if own_idx is None:
    refs.append(row["expr"]); own_idx = len(refs) - 1

prompts     = [prompt] * len(refs)
completions = [row["completion"]] * len(refs)
print(f"step={step} idx={idx} correct={row['correct']} expr={row['expr']!r} "
      f"→ {len(refs)} refs (own_idx={own_idx})")


/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


KeyError: 'completion'

In [72]:
from src.velocity import compute_vt_batched

# ============================
# Velocity Reward Computation
# ============================

scored = compute_vt_batched(prompts, completions, refs, scorer, tokenizer,
                            micro_batch_size=8)
 

KeyboardInterrupt: 

In [ ]:
# ===================================================
# Benchmark v_t reward computation across configs
# ===================================================
# Compares wall-time and numerical agreement (R_T) for combinations of
# (micro_batch_size, chunk_size). Use to sanity-check that a faster setting
# does not perturb R_T more than expected.
#
# Inputs: `prompts`, `completions`, `refs` from the cell above; `scorer`
# loaded on GPU. The cell does its own CUDA sync to get honest wall times.
import time
import numpy as np
import torch
from src.velocity import compute_vt_batched

assert "scorer" in globals(), "load `scorer` first (cell 1 model-load block)"

def _bench(label, *, micro_batch_size, chunk_size, warmup=False):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    scored = compute_vt_batched(
        prompts, completions, refs, scorer, tokenizer,
        micro_batch_size=micro_batch_size,
        chunk_size=chunk_size,
    )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = time.time() - t0
    R_Ts = np.array([s["R_T"] for s in scored], dtype=float)
    if not warmup:
        peak_gb = (torch.cuda.max_memory_allocated() / 1e9
                   if torch.cuda.is_available() else 0.0)
        print(f"  {label:38s}  t={dt:7.2f}s  "
              f"peak={peak_gb:5.1f}GB  "
              f"R_T mean={R_Ts.mean():+.3f}  std={R_Ts.std():.3f}  "
              f"n_refs={len(scored)}  vt_len={len(scored[0]['vt'])}")
    return dt, R_Ts, scored

# Warmup (capture graphs, allocate workspace, prime caches)
print("[warmup] one short run to prime caches ...")
_bench("warmup-discard", micro_batch_size=8, chunk_size=64, warmup=True)

print("\n[bench] micro_batch_size sweep at chunk_size=1 (baseline grid)")
results_mb = {}
for mb in (8, 32, 64, 128, 256, 512):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    try:
        dt, R_Ts, _ = _bench(f"micro_batch={mb:4d}, chunk=1", micro_batch_size=mb, chunk_size=1)
        results_mb[mb] = (dt, R_Ts)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f"  micro_batch={mb:4d}  OOM — that's your ceiling")
        break

print("\n[bench] chunk_size sweep at micro_batch=128")
results_ck = {}
ref_R = None
for ck in (1, 2, 4, 8, 16, 32):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    dt, R_Ts, _ = _bench(f"chunk={ck:3d}, micro_batch=128", micro_batch_size=128, chunk_size=ck)
    results_ck[ck] = (dt, R_Ts)
    if ck == 1:
        ref_R = R_Ts
    else:
        delta = np.abs(R_Ts - ref_R).max()
        print(f"    max |R_T(ck={ck}) - R_T(ck=1)| = {delta:.2e}")

# Summary table -------------------------------------------------------------
if results_ck:
    base = results_ck.get(1, (None,))[0]
    print("\n[summary] chunk_size speedup vs ck=1 (micro_batch=128)")
    for ck, (dt, _) in results_ck.items():
        if base:
            print(f"  ck={ck:3d}  {dt:6.2f}s   speedup={base/dt:5.2f}x")


In [ ]:
# ===================================================
# Benchmark: torch.compile the scorer
# ===================================================
# Wraps `scorer.forward` with torch.compile and re-runs the chunk sweep,
# comparing wall-time and R_T against the uncompiled baseline. The first
# call after compile() is expensive (graph capture + autotune); we discard
# the first run and report only steady-state numbers.
#
# Notes:
# - mode="reduce-overhead" prioritizes kernel-launch reduction (what we
#   want, since the rest-forward is launch-bound on short answers).
# - dynamic=True keeps graphs reusable across varying (B, La, Cmax) shapes.
#   Without it, every distinct (B, La, Cmax) triggers a recompile which
#   destroys the win on variable-length input.
# - If compile fails or recompiles thrash, fall back gracefully.
import time
import numpy as np
import torch
from src.velocity import compute_vt_batched

assert "scorer" in globals(), "load `scorer` first"

# Establish uncompiled baseline (one config, several reps for variance).
print("[baseline] uncompiled, chunk=8, micro_batch=128")
baseline_times = []
for rep in range(3):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    scored_base = compute_vt_batched(
        prompts, completions, refs, scorer, tokenizer,
        micro_batch_size=128, chunk_size=8,
    )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    baseline_times.append(time.time() - t0)
    print(f"  rep {rep}: {baseline_times[-1]:.2f}s")
R_T_base = np.array([s["R_T"] for s in scored_base])

# Compile.
print("\n[compile] wrapping scorer with torch.compile(mode='reduce-overhead', dynamic=True) ...")
try:
    # We compile a non-destructive proxy so the global `scorer` is unchanged
    # for downstream notebook cells.
    scorer_compiled = torch.compile(
        scorer, mode="reduce-overhead", dynamic=True, fullgraph=False,
    )
    # Warmup: 2 calls to absorb compilation + autotune; discard timings.
    for w in range(2):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        _ = compute_vt_batched(
            prompts, completions, refs, scorer_compiled, tokenizer,
            micro_batch_size=128, chunk_size=8,
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        print(f"  warmup {w}: {time.time()-t0:.2f}s (capture / autotune)")

    print("\n[compiled] steady-state, chunk=8, micro_batch=128")
    compiled_times = []
    for rep in range(3):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.time()
        scored_c = compute_vt_batched(
            prompts, completions, refs, scorer_compiled, tokenizer,
            micro_batch_size=128, chunk_size=8,
        )
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        compiled_times.append(time.time() - t0)
        print(f"  rep {rep}: {compiled_times[-1]:.2f}s")
    R_T_c = np.array([s["R_T"] for s in scored_c])

    # Verdict ---------------------------------------------------------------
    b_med = float(np.median(baseline_times))
    c_med = float(np.median(compiled_times))
    max_dR = float(np.max(np.abs(R_T_base - R_T_c)))
    print(f"\n[verdict] median baseline {b_med:.2f}s  →  compiled {c_med:.2f}s  "
          f"speedup={b_med/c_med:.2f}×")
    print(f"          max |R_T(compiled) - R_T(uncompiled)| = {max_dR:.2e}")
    if max_dR > 1e-2:
        print("  ⚠ numerical drift > 1e-2 — investigate before enabling in prod")
    elif c_med >= 0.95 * b_med:
        print("  → not worth keeping (no speedup or slower); use uncompiled")
    else:
        print(f"  → keep compile: ~{(1 - c_med/b_med)*100:.0f}% wall-clock saving")
except Exception as e:
    print(f"  compile path raised {e.__class__.__name__}: {e}")
    print("  → fall back to uncompiled")


In [ ]:
# ===================================================
# Sanity check: chunk_size invariance + grouping correctness
# ===================================================
# Two correctness claims for the optimizations in src/velocity.py:
#
#   (A) R_T is exact under any chunk_size (endpoints 0 and T always included,
#       and logps[-1] - logps[0] is independent of intermediate sampling).
#   (B) Grouping refs by shared (q_ids, o_ids) does not change per-ref outputs.
#       The prefix forward is deterministic for fixed (q, o); reusing its cache
#       across refs is mathematically identical to recomputing it per ref.
#
# This cell verifies both empirically. Use it after any future change that
# touches the prefix-sharing or chunking logic, before re-running the sweep.
import numpy as np
import torch
from src.velocity import compute_vt_batched

assert "scorer" in globals(), "load `scorer` first"

# (A) chunk_size invariance: R_T must match across all chunk sizes.
print("[sanity-A] R_T invariance over chunk_size")
ref = compute_vt_batched(
    prompts, completions, refs, scorer, tokenizer,
    micro_batch_size=128, chunk_size=1,
)
R_T_ref = np.array([s["R_T"] for s in ref])
for ck in (2, 4, 8, 16, 32):
    out = compute_vt_batched(
        prompts, completions, refs, scorer, tokenizer,
        micro_batch_size=128, chunk_size=ck,
    )
    R_T = np.array([s["R_T"] for s in out])
    max_d = float(np.max(np.abs(R_T - R_T_ref)))
    status = "OK" if max_d < 1e-3 else "DRIFT"
    print(f"  ck={ck:3d}  max|ΔR_T|={max_d:.2e}  [{status}]")

# (B) prefix-sharing correctness: scoring all refs at once must match scoring
# one ref at a time (where grouping has nothing to share).
print("\n[sanity-B] grouped vs per-ref invariance")
out_grouped = compute_vt_batched(
    prompts, completions, refs, scorer, tokenizer,
    micro_batch_size=128, chunk_size=1,
)
R_T_grouped = np.array([s["R_T"] for s in out_grouped])

R_T_solo = np.empty(len(refs), dtype=float)
for i, r in enumerate(refs):
    o = compute_vt_batched(
        [prompts[i]], [completions[i]], [r], scorer, tokenizer,
        micro_batch_size=128, chunk_size=1,
    )
    R_T_solo[i] = o[0]["R_T"]
max_d = float(np.max(np.abs(R_T_grouped - R_T_solo)))
status = "OK" if max_d < 1e-3 else "DRIFT"
print(f"  max|R_T_grouped - R_T_solo| = {max_d:.2e}  [{status}]")


In [ ]:
# ====================================
# Visualize Velocity Reward across solutions
# (left: cum R(t) per ref, dotted ¶ boundaries; right: final R_T sorted)
# ====================================
from src.velo_viz import prepare, plot_R_t_static
from src.velo_viz import print_paragraphs
from src.velo_viz import make_animation
import matplotlib.pyplot as plt

ctx = prepare(row, scored, refs, own_idx, tokenizer)
# plot_R_t_static(ctx); plt.show()
# print_paragraphs(ctx)
make_animation(ctx, save_path="logs/r_t_stream.gif", fps=10)

NameError: name 'top_mistake' is not defined